In [25]:
import os
import time
import threading
import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Set your Google Gemini API Key
os.environ["GOOGLE_API_KEY"] = "API_KEY"

# 2. Initialize FastAPI app with the new name
app = FastAPI(title="XYZ Finance Voice Assistant - Colab")

# 3. Initialize Google Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7
)

# 4. Updated FAQ Knowledge Base Data for XYZ Finance
faq_data = [
  {
    "question": "What are the interest rates for XYZ Finance Personal Loans?",
    "answer": "XYZ Finance Digital Personal Loans offer interest rates starting from 11.5% per annum, with loan amounts up to Rs. 30 Lakhs and repayment tenures up to 60 months."
  },
  {
    "question": "How fast can I get a Two-Wheeler Loan approved?",
    "answer": "XYZ Finance offers instant paperless approval for Two-Wheeler Loans within 8 minutes using digital KYC."
  },
  {
    "question": "How do I download my Loan Account Statement or NOC?",
    "answer": "You can instantly download your Statement of Account (SOA) or No Objection Certificate (NOC) using the XYZ Finance PLANET App or by logging into the customer portal at xyzfinance.com."
  },
  {
    "question": "What happens if I miss my EMI payment date?",
    "answer": "Missing an EMI incurs a bounce charge along with penal interest. You can make an immediate self-service payment via the PLANET App or through our QuickPay link to avoid impact on your CIBIL score."
  }
]

# 5. Request Schema
class VoiceInteractionRequest(BaseModel):
    message: str

# 6. Chatbot API Endpoint
@app.post("/api/v1/voice-interaction")
async def voice_interaction(request: VoiceInteractionRequest):
    try:
        faq_context = "Here is the official FAQ information for XYZ Finance:\n"
        for item in faq_data:
            faq_context += f"Q: {item['question']}\nA: {item['answer']}\n\n"

        system_prompt = (
            "You are a helpful, professional multilingual AI Voice Assistant for XYZ Finance. "
            "Use the following FAQ knowledge base to answer user queries accurately:\n\n"
            f"{faq_context}"
        )

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=request.message)
        ]

        response = llm.invoke(messages)

        return {
            "status": "success",
            "user_input": request.message,
            "bot_output": response.content
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 7. Start FastAPI in a background thread
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)  # Give the server 2 seconds to start up

# 8. Interactive Chat Loop
print("✅ XYZ Finance Chatbot is live! Type 'exit' to stop.\n")

while True:
    user_query = input("You: ")
    if user_query.strip().lower() in ["exit", "quit"]:
        print("Chat ended.")
        break
    if not user_query.strip():
        continue

    res = requests.post(
        "http://127.0.0.1:8000/api/v1/voice-interaction",
        json={"message": user_query}
    )

    if res.status_code == 200:
        data = res.json()
        print(f"Bot: {data['bot_output']}\n" + "-"*50)
    else:
        print(f"Error ({res.status_code}): {res.text}\n")

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


✅ XYZ Finance Chatbot is live! Type 'exit' to stop.

You: HII


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Bot: [{'type': 'text', 'text': 'Hello! Welcome to L&T Finance. How can I assist you with your loan or financial needs today?', 'extras': {'signature': 'EmAKXgFpFH0TS+C3enOSFv+kh9+iQxX4+/rs3b0SH20V+dLjQnBoMZPGi2h13GRYjuNad1QqYsRw8BEGn67QPwSGlOp4WDMrwV3szzn8AZfEapHODkoNmkJrRJn2Hmd46Q4='}}]
--------------------------------------------------
You: HOW ARE YOU


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Bot: [{'type': 'text', 'text': 'Hello! I am doing well, thank you for asking. Welcome to L&T Finance! How can I assist you with your loan or financial queries today?', 'extras': {'signature': 'EmAKXgFpFH0TC97ttS/QKG0p5j7V3A6vGeNhbX0SFYQCIdWooBR/QizzaoW2vnWWGnsKSiDT3sxaPr+ukg2IFzboWmxaXQaCYKhwSL7tyD7h/urhhb0Xwx/YTn/IzMCU2Us='}}]
--------------------------------------------------


KeyboardInterrupt: Interrupted by user